# T1 Introduction: The Manifold Learning Pipeline

This tutorial is the `topometryNoSC` equivalent of the original [TopoMetry Introduction](https://topometry.readthedocs.io/en/latest/T1_introduction.html). Because `topometryNoSC` is a pure geometry engine without the single-cell dependencies (`scanpy`/`AnnData`), we simply run the identical mathematical pipeline using standard `scikit-learn` array semantics.

### Data Provenance
We use a pre-processed and sub-sampled version of the PBMC68k dataset. The preprocessing steps (QC, Z-score, HVGs) have been performed offline. Please see `docs/tutorials/data-provenance.md` for the exact script used to generate this dataset.

Let's import `topometryNoSC` and our plotting/evaluation utilities to keep things pretty!

In [ ]:
import os
import sys

from _example_utils import (
    DemoConfig,
    DemoData,
    compute_metrics,
    plot_metric_overview,
    print_metric_summary,
    run_pipeline,
)

import topo as tp

sys.path.append(os.path.abspath("../docs/tutorials"))
from data import load_cells

## 1. Load the Data
We load our expression matrix `X` and our group annotations `labels`. The engine only operates on `X`; the `labels` are used strictly for colouring the generated layouts.

In [ ]:
X, labels, label_names = load_cells(return_names=True)
print(f"Data matrix shape: {X.shape}")
print(f"Number of unique cell types: {len(set(labels))}")

## 2. The Automated Pipeline
In the original tutorial, you'd run `tp.sc.run_and_report()`. Here, we just instantiate `TopOGraph` and call `fit(X)`. This single call handles:
1. **Neighborhood graphs** (CkNN or adaptive bandwidth)
2. **Diffusion operators**
3. **Intrinsic Dimensionality (ID)** estimation
4. **Spectral scaffolds** (msDM / DM)
5. **Refined graphs** and **2D Layouts** (TopoMAP, TopoPaCMAP)

In [ ]:
tg = tp.TopOGraph(random_state=42)
tg.fit(X)

## 3. Intrinsic Dimensionality (ID)
The dataset's effective complexity determines the scale of the spectral scaffold.

In [ ]:
print(f"Estimated Global ID: {tg.global_id}")
print("ID Method:", tg.intrinsic_dim["method"])

## 4. Visualization & Layouts
We can easily plot the results of our pipeline. `topometryNoSC` provides its own plotting utility inside `topo.plot` that accepts any generated layout.

In [ ]:
fig = tp.plot.scatter(tg.msTopoMAP, labels=labels, pt_size=6)
fig.suptitle("msTopoMAP: Multiscale Diffusion Map Layout", fontsize=14, y=1.02);

## 5. Denoising / Imputation
Technical artifacts (like dropout) can be mitigated by diffusing signals across the spectral scaffold. We can filter the entire array `X`.

In [ ]:
# Diffuse the data over the spectral scaffold for 't' steps
X_imputed = tg.impute(X, t=3, which="msZ")
print(f"Imputed data shape: {X_imputed.shape}")

## 6. Geometry-Preservation Metrics & Utilities overview
To fully replicate the deep visual reporting previously provided by `run_and_report`, we can make use of `_example_utils.py` to calculate operator-native metrics (like Neighborhood PF1, Spectral Procrustes, Geodesic Rank Correlation) and visualize the manifold learning pipeline end-to-end!

In [ ]:
# Wrap our loaded data for the utils
data = DemoData(X=X, color=labels)

# Setup a configuration using the PaCMAP layout and multiscale Diffusion Maps (msDM)
config = DemoConfig(projection_method="PaCMAP", dm_method="msDM")

# Run the extended pipeline using utils
result = run_pipeline(data, config)

# Compute all metrics natively
metrics = compute_metrics(data, result, config)

# Print out the metrics summary
print_metric_summary(metrics)

In [ ]:
# Plot the comprehensive metric overview
plot_metric_overview(data, result, metrics, config)